# NEM demand from `NEMOSIS`

#### Retrieve demand data from AEMO data portal (as required)

In [4]:
# from nemosis import dynamic_data_compiler

In [5]:
#e.g. for 2009-2024 data
# d = dynamic_data_compiler("2009/01/01 00:00:00", "2024/12/31 23:00:00", "DISPATCHREGIONSUM", "/g/data/w42/dr6273/tmp", select_columns=["SETTLEMENTDATE", "REGIONID", "TOTALDEMAND"])

### Open demand data

"DISPATCHREGIONSUM" data, which are at 5-minute intervals.

In [6]:
import pandas as pd
import glob

In [7]:
data_path = "/g/data/w42/dr6273/tmp/"

In [45]:
files = sorted(glob.glob(data_path+"*DISPATCHREGIONSUM*.CSV"))

In [46]:
files[:3] # More recent dates have a different file name e.g. Aug-Dec 2024

['/g/data/w42/dr6273/tmp/PUBLIC_ARCHIVE#DISPATCHREGIONSUM#FILE01#202408010000.CSV',
 '/g/data/w42/dr6273/tmp/PUBLIC_ARCHIVE#DISPATCHREGIONSUM#FILE01#202409010000.CSV',
 '/g/data/w42/dr6273/tmp/PUBLIC_ARCHIVE#DISPATCHREGIONSUM#FILE01#202410010000.CSV']

In [47]:
files[-3:]

['/g/data/w42/dr6273/tmp/PUBLIC_DVD_DISPATCHREGIONSUM_202405010000.CSV',
 '/g/data/w42/dr6273/tmp/PUBLIC_DVD_DISPATCHREGIONSUM_202406010000.CSV',
 '/g/data/w42/dr6273/tmp/PUBLIC_DVD_DISPATCHREGIONSUM_202407010000.CSV']

In [48]:
def open_df(path):
    return pd.read_csv(
        path,
        header=1,
        usecols=["REGIONID", "SETTLEMENTDATE", "TOTALDEMAND"],
        parse_dates=["SETTLEMENTDATE"],
        index_col="SETTLEMENTDATE",
    )

In [49]:
dfs = [open_df(f) for f in files]

In [50]:
df = pd.concat(dfs)

In [51]:
df = df.sort_index()

In [52]:
df.head()

,REGIONID,TOTALDEMAND
SETTLEMENTDATE,,
2009-07-01 00:05:00,NSW1,8770.64
2009-07-01 00:05:00,VIC1,5405.90
2009-07-01 00:05:00,QLD1,5219.07
2009-07-01 00:05:00,SA1,1591.33
2009-07-01 00:05:00,TAS1,933.98


In [53]:
df.tail()

,REGIONID,TOTALDEMAND
SETTLEMENTDATE,,
NaT,NaN,NaN
NaT,NaN,NaN
NaT,NaN,NaN
NaT,NaN,NaN
NaT,NaN,NaN


In [54]:
df.shape

(8435896, 2)

Remove NaNs

In [55]:
df.loc[df.REGIONID.isnull()].head()

,REGIONID,TOTALDEMAND
SETTLEMENTDATE,,
NaT,NaN,NaN
NaT,NaN,NaN
NaT,NaN,NaN
NaT,NaN,NaN
NaT,NaN,NaN


In [56]:
df = df.loc[~df.REGIONID.isnull()]

In [57]:
df.shape

(8435710, 2)

In [58]:
len(pd.date_range("2009-07-01 00:05:00", "2025-01-01 00:00:00", freq="5min")) * 5

8154720

Lengths don't match because we have some duplicate index values. Don't remove now as need to retain each region

In [59]:
len(pd.date_range("2009-07-01 00:05:00", "2025-01-01 00:00:00", freq="5min"))

1630944

In [60]:
len(pd.date_range("2009-07-01 00:05:00", "2025-01-01 00:00:00", freq="30min"))

271824

### Process each region separately

In [61]:
regions = pd.unique(df.REGIONID)
regions

array(['NSW1', 'VIC1', 'QLD1', 'SA1', 'TAS1'], dtype=object)

In [62]:
dfs = []
for r in regions:
    rdf = df.loc[df.REGIONID == r]
    rdf = rdf.loc[~rdf.index.duplicated()]
    rdf = rdf.drop("REGIONID", axis=1)
    rdf.columns = [r]
    dfs.append(rdf)

In [63]:
dem = pd.concat(dfs, axis=1)
dem

,NSW1,VIC1,QLD1,SA1,TAS1
SETTLEMENTDATE,,,,,
2009-07-01 00:05:00,8770.64,5405.90,5219.07,1591.33,933.98
2009-07-01 00:10:00,8808.24,5368.79,5271.33,1589.66,934.70
2009-07-01 00:15:00,8823.20,5362.76,5191.56,1609.87,929.88
2009-07-01 00:20:00,8776.82,5311.96,5154.33,1594.49,924.12
2009-07-01 00:25:00,8725.02,5301.42,5149.72,1582.01,921.72
...,...,...,...,...,...
2024-12-31 23:40:00,7467.70,4441.76,6497.05,1375.14,986.22
2024-12-31 23:45:00,7393.27,4373.63,6573.14,1369.37,987.45
2024-12-31 23:50:00,7360.53,4379.26,6567.08,1372.97,983.12


Convert from AEST to UTC to align with BARRA-C2

In [64]:
dem.index -= pd.DateOffset(hours=10)

The data are in MW. This is the output at any moment. We want to convert this to MWh. To convert these 5-minute ratings, we divide them by (60 / 5) = 12:

In [65]:
dem_mwh = dem / 12

Then we can sum by hour

In [66]:
dem_hourly = dem_mwh.resample("h").sum()

In [67]:
dem_hourly

,NSW1,VIC1,QLD1,SA1,TAS1
SETTLEMENTDATE,,,,,
2009-06-30 14:00:00,7945.377500,4810.673333,4679.339167,1438.372500,845.318333
2009-06-30 15:00:00,8285.597500,5393.625833,4746.761667,1343.127500,904.636667
2009-06-30 16:00:00,7586.278333,5038.578333,4505.274167,1182.105000,895.562500
2009-06-30 17:00:00,7039.883333,4771.565833,4434.766667,1048.341667,906.427500
2009-06-30 18:00:00,6915.753333,4660.491667,4525.978333,1046.805833,948.933333
...,...,...,...,...,...
2024-12-31 10:00:00,7926.642500,4692.180833,7384.685833,1536.267500,1050.976667
2024-12-31 11:00:00,7834.642500,4513.946667,7188.300000,1452.518333,1038.005833
2024-12-31 12:00:00,7681.166667,4343.268333,6881.964167,1372.003333,1009.574167


Remove first and last day as incomplete days

In [68]:
dem_hourly = dem_hourly.loc["2009-07-01" : "2024-12-30"]

In [69]:
dem_hourly

,NSW1,VIC1,QLD1,SA1,TAS1
SETTLEMENTDATE,,,,,
2009-07-01 00:00:00,9811.083333,6737.801667,6319.625000,1694.600833,1359.704167
2009-07-01 01:00:00,9518.105000,6679.405000,6201.692500,1715.465000,1293.429167
2009-07-01 02:00:00,9362.036667,6602.611667,6183.130833,1686.535833,1279.117500
2009-07-01 03:00:00,9228.744167,6730.846667,6156.053333,1695.395833,1265.324167
2009-07-01 04:00:00,9131.188333,6599.993333,6155.266667,1677.635833,1301.748333
...,...,...,...,...,...
2024-12-30 19:00:00,6518.968333,4034.800833,5956.408333,1296.312500,1088.774167
2024-12-30 20:00:00,6443.495833,3984.287500,5971.838333,1236.905833,1079.060000
2024-12-30 21:00:00,6501.412500,3704.450833,5908.420000,1026.789167,1047.042500


Rename columns

In [70]:
dem_hourly.columns = ["NSW", "QLD", "SA", "TAS", "VIC"]

Write

In [75]:
dem_hourly.to_csv(
    "/g/data/w42/dr6273/work/projects/Aus_energy/data/energy_demand/hourly_demand_20090701-20241230.csv"
)